# পাঠ ০২ - মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক অন্বেষণ

**মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক (MAF)** হল AI এজেন্ট তৈরি করার জন্য একটি একীভূত ফ্রেমওয়ার্ক। এটি চারটি মূল নির্মাণ ব্লকসহ একটি পরিষ্কার, কম্পোজেবল আর্কিটেকচার প্রদান করে:

- **ক্লায়েন্ট** – একটি AI মডেল এন্ডপয়েন্টের সাথে সংযুক্ত হয় এবং যোগাযোগ পরিচালনা করে
- **এজেন্ট** – ক্লায়েন্টকে নির্দেশাবলী এবং টুল সংজ্ঞা সহ আবৃত করে
- **টুলস** – এজেন্টের ক্ষমতা বাড়ায় কাস্টম ফাংশন দিয়ে যা মডেল কল করতে পারে
- **সেশন** – বহু-টান কথোপকথনের জন্য কথোপকথন ইতিহাস বজায় রাখে

এই পাঠে, আমরা এই ধারণাগুলি ব্যবহার করে একটি **ভ্রমণ বুকিং এজেন্ট** তৈরি করব যা গন্তব্যের সুলভতা পরীক্ষা করে।


## সেটআপ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## এজেন্ট ফ্রেমওয়ার্ক আর্কিটেকচার বোঝা

Microsoft এজেন্ট ফ্রেমওয়ার্ক একটি স্তরবিশিষ্ট আর্কিটেকচার অনুসরণ করে:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ক্লায়েন্ট** – একটি `FoundryChatClient` একটি Azure OpenAI ডিপ্লয়মেন্টের সাথে সংযুক্ত হয়। এটি প্রমাণীকরণ, অনুরোধ ফরম্যাটিং এবং প্রতিক্রিয়া বিশ্লেষণ পরিচালনা করে।
2. **এজেন্ট** – ক্লায়েন্ট থেকে `provider.create_agent()` এর মাধ্যমে তৈরি, এজেন্ট মডেল অ্যাক্সেসকে নির্দেশনা (সিস্টেম প্রম্পট) এবং টুলসের সাথে সংযুক্ত করে।
3. **টুলস** – `@tool` দিয়ে সজ্জিত পাইথন ফাংশন যা এজেন্ট কাজ করার জন্য অথবা তথ্য উৎসাহিত করার জন্য কল করতে পারে।
4. **সেশন** – একটি `AgentSession` অবজেক্ট (যা `agent.create_session()` থেকে তৈরি) যেটি সংলাপের ইতিহাস সঞ্চয় করে, যা বহু-বার্তার কথোপকথন সক্ষম করে যেখানে এজেন্ট পূর্বের প্রসঙ্গ মনে রাখে।

আসুন প্রতিটি স্তর ক্রমে তৈরি করি।


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool ডেকোরেটর সহ টুল যোগ করা

টুলগুলি এজেন্টদের শুধুমাত্র টেক্সট তৈরি ছাড়িয়ে কর্ম নিতে দেয়। `@tool` ডেকোরেটর একটি সাধারণ পাইথন ফাংশনকে এমন কিছুতে রূপান্তর করে যা এজেন্ট কল করতে পারে।

মূল বিষয়বস্তুসমূহ:
- মডেল প্রতিটি প্যারামিটার বুঝতে `Annotated[type, "description"]` ব্যবহার করুন।
- ডকস্ট্রিং হয়ে ওঠে টুলের বর্ণনা যা মডেল দেখবে।
- `approval_mode="never_require"` অর্থ টুলটি স্বয়ংক্রিয়ভাবে চালু হয় ব্যবহারকারীর অনুমতি ছাড়াই।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## টুল সহ এজেন্ট তৈরি করা

এখন আমরা ক্লায়েন্ট, নির্দেশাবলী, এবং টুলগুলোকে একসাথে করে একটি এজেন্ট তৈরি করব। `instructions` সিস্টেম প্রম্পট হিসেবে কাজ করে — এগুলো এজেন্টের ব্যক্তিত্ব এবং আচরণ নির্ধারণ করে।


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## সেশন সহ বহু-পর্যায় কথোপকথন

একটি `AgentSession` (`agent.create_session()` এর মাধ্যমে তৈরি) কথোপকথনের সমস্ত মেসেজ ট্র্যাক করে। একই সেশন প্রতিটি `agent.run()` কল-এ পাস করে, এজেন্ট সম্পূর্ণ কথোপকথনের ইতিহাস অ্যাক্সেস করতে পারে এবং পূর্বের মেসেজগুলির দিকে ফিরে যেতে পারে।

আমরা `tools=[check_destination_availability]` পাস করি যাতে এজেন্ট প্রতিটি পর্যায়ে আমাদের উপলব্ধতা পরীক্ষক কল করতে পারে।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## সারসংক্ষেপ

এই পাঠে আপনি Microsoft Agent Framework-এর চারটি স্তম্ভ অন্বেষণ করেছেন:

| ধারণা | আপনি যা শিখেছেন |
|---------|------------------|
| **ক্লায়েন্ট** | `FoundryChatClient` ক্রেডেনশিয়াল-ভিত্তিক প্রমাণীকরণের মাধ্যমে Azure OpenAI-তে সংযুক্ত হয় |
| **এজেন্ট** | `provider.create_agent()` একটি মডেল সংযোগকে নির্দেশাবলী এবং একটি নামের সাথে বান্ডিল করে |
| **টুলস** | `@tool` ডেকোরেটর Python ফাংশনসমূহকে প্রকাশ করে যাতে এজেন্ট কল করতে পারে |
| **সেশন** | `agent.create_session()` একাধিক পালার মধ্যে কথোপকথনের ইতিহাস রক্ষা করে |

এই নির্মাণ ব্লকগুলো একত্রিত হয়ে এমন এজেন্ট তৈরি করে যা স্বাভাবিক কথোপকথন রাখতে পারে, বাইরের ফাংশন কল করতে পারে এবং প্রাসঙ্গিকতা বজায় রাখতে পারে — যা পরবর্তী পাঠগুলিতে আরও উন্নত এজেন্টিক প্যাটার্নের ভিত্তি। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
